# Bronze Layer - Raw Data Ingestion

## Purpose 
Ingests raw sales data and writes it to the Bronze Delta table with no transformations applied.

## Reads from
Raw data defined in this notebook (simulating an external source)

## Writes to
- 'pipeline_bronze' - raw sales data as Delta table

## Notes
- No cleaning or transformation happens here
- All data is written exactly as recieved
- Row count is logged for validation

In [0]:
dbutils.widgets.text("run_date", "2026-01-01", "Run Date")
dbutils.widgets.text("env", "dev", "Environment")

run_date = dbutils.widgets.get("run_date")
env = dbutils.widgets.get("env")

print(f"Run date: {run_date}")
print(f"Environment: {env}")

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)
from pyspark.sql.functions import col, lit, current_timestamp
from datetime import datetime

TABLE_NAME = f"pipeline_bronze"
ENV        = env
RUN_DATE   = run_date

print(f"Config loaded — table: {TABLE_NAME}, env: {ENV}")

In [0]:
schema = StructType([
    StructField("order_id",    IntegerType(), True),
    StructField("customer",    StringType(),  True),
    StructField("city",        StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("amount",      DoubleType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("order_date",  StringType(),  True),
    StructField("rep",         StringType(),  True),
])

raw_data = [
    (1,  "Alice",   "chicago",   "electronics", 1200.50, 1, "2024-01-15", "Bob"),
    (2,  "Bob",     "New York",  "clothing",      89.99, 2, "2024-01-15", "Sara"),
    (3,  "Carol",   "CHICAGO",   "electronics",  450.00, 1, "2024-01-16", "Bob"),
    (4,  "David",   "houston",   "food",           23.49, 3, "2024-01-16", "Mike"),
    (5,  "Eve",     "New York",  "electronics",  999.99, 1, "2024-01-17", "Sara"),
    (6,  "Frank",   "Chicago",   "clothing",     199.00, 2, "2024-01-17", "Bob"),
    (7,  "Grace",   "houston",   "food",           55.00, 4, "2024-01-18", "Mike"),
    (8,  "Henry",   "New York",  "electronics", 2300.00, 1, "2024-01-18", "Sara"),
    (9,  "Ivy",     "Chicago",   "food",           18.75, 2, "2024-01-19", "Bob"),
    (10, "James",   "houston",   "clothing",     349.00, 1, "2024-01-19", "Mike"),
    (11, "Karen",   "Chicago",   "electronics",  870.00, 1, "2024-01-20", "Bob"),
    (12, "Leo",     "New York",   None,           125.00, 2, "2024-01-20", "Sara"),
    (13, "Maya",    "Chicago",   "food",           67.50, 1, "2024-01-21", "Bob"),
    (14, "Nathan",  "houston",   "electronics",  540.00, 1, "2024-01-21", "Mike"),
    (15, "Olivia",  "New York",  "clothing",     210.00, 2, "2024-01-22", "Sara"),
]

df_raw = spark.createDataFrame(raw_data, schema)
print(f"Raw data loaded — {df_raw.count()} rows")

In [0]:
df_bronze = df_raw \
    .withColumn("ingested_at", current_timestamp()) \
    .withColumn("run_date",    lit(RUN_DATE)) \
    .withColumn("env",         lit(ENV))

df_bronze.show(5)
print(f"Schema after metadata columns:")
df_bronze.printSchema()

In [0]:
row_count    = df_bronze.count()
null_amounts = df_bronze.filter(col("amount").isNull()).count()
null_cats    = df_bronze.filter(col("category").isNull()).count()

print(f"Row count:        {row_count}")
print(f"Null amounts:     {null_amounts}")
print(f"Null categories:  {null_cats}")

if row_count == 0:
    raise Exception("BRONZE FAILED: No rows to write — aborting pipeline")

if null_amounts > 0:
    print(f"WARNING: {null_amounts} rows have null amounts — investigate")

print("Validation passed — proceeding to write")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLE_NAME)

final_count = spark.read.table(TABLE_NAME).count()
print(f"Bronze write complete — {final_count} rows in {TABLE_NAME}")
print(f"Run date: {RUN_DATE} | Env: {ENV} | Time: {datetime.now()}")